In [1]:
# Load environment variables and imports
from dotenv import load_dotenv
load_dotenv()
import os

In [2]:
# Create the OpenAI client from the environment variable
from openai import OpenAI
openai_client = OpenAI(api_key=os.getenv("OPEN_API_KEY"))

In [3]:
# Define a helper to call the model
def llm(prompt):
    response = openai_client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [4]:
# No context here: the model answers generically and may ask for the specific course
question = "I just discovered this course. Can I join now ?"
answer = llm(question)
print(answer)

Could you please provide the name or link of the course you're referring to? That way, I can give you accurate information about enrollment.


In [5]:
# Set a simple FAQ context for basic RAG
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [7]:
# Build the RAG prompt by combining the question with the provided context
prompt = f"""
Your task is to answer questions from the course participants based on the provided context.
Use the context to find relevant information and provide accurate and helpful answers to the questions. 
If the context does not contain enough information to answer a question, respond with "I don't know".

Question:
{question}

Context:
{context}
"""

In [8]:
# Ask the model and print the answer
answer = llm(prompt)
print(answer)

Yes, you can join the course now. You can start learning and submitting homework even if you just discovered the course. However, if you want to receive a certificate, make sure to submit your project while submissions are still being accepted.


In [9]:
# Fetch the course FAQ JSON from the website
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()
print(courses_raw)

[{'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 404}, {'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}, {'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 41}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 85}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 255}, {'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 472}]


In [35]:
# Build a list of course FAQ documents
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course['path']}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()  # Raise an error if the request was unsuccessful
    course_data = course_response.json()
    documents.extend(course_data)

print(len(documents))

1350


In [11]:
# Each entry has:

#     id - unique identifier
#     course - course slug (e.g., machine-learning-zoomcamp)
#     section - which section of the course
#     question - the FAQ question
#     answer - the FAQ answer


documents[0]

# Each course has a slug - a short identifier used in URLs. For example, machine-learning-zoomcamp, data-engineering-zoomcamp, etc. 
# We'll use these slugs for filtering in search.  For example, if a student asks about the data engineering course, we skip results from the ML course

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [ ]:
# 1. Search

# We can use the MinSearch library to create a simple search index for our course FAQ documents.
# MinSearch allows us to define which fields are used for full-text search and which fields are used for keyword filtering.

from minsearch import Index

index = Index(
    text_fields=["question", "answer", "section"],
    keyword_fields=["course"]
)

index.fit(documents) 



In [ ]:
# RAG starts with search

def search(question, course='llm-zoomcamp'):
    boost_dict={"question": 2.0, "section": 0.5} # Boost the question field twice as much as the section field
    filter_dict={"course": course}

    return index.search(
        question, 
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [21]:
search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [26]:
# 2. Build context

# Prompt has two parts: instructions and the question + context. We can separate them for better readability and maintenance.
# Instructions are general guidelines for the model on how to use the context to answer questions. The question + context part is dynamic and changes with each user query.

INSTRUCTIONS = """
Your task is to answer questions from the course participants based on the provided context.
Use the context to find relevant information and provide accurate and helpful answers to the questions. 
If the context does not contain enough information to answer a question, respond with "I don't know".
"""

In [27]:
# Build the RAG prompt by combining the question with the provided context

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [23]:
# We can also build the context in a more structured way, for example, by including section titles and formatting the question-answer pairs clearly. 
# This can help the model better understand the information and improve the quality of the answers.

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(f"Section: {doc['section']}")
        lines.append(f"Q: {doc['question']}")
        lines.append(f"A: {doc['answer']}")
        lines.append("")  # Add an empty line for better readability

    return "\n".join(lines)

In [29]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(question=question, context=context) 
    return prompt.strip()  # Remove leading/trailing whitespace

In [32]:
prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered this course. Can I join now ?

Context:
Section: General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

Section: General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

Section: General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) 

In [ ]:
# 3. Ask the model

# OpenAI has two APIs for interacting with models: the Chat Completions API and the Responses API. 
# The Completions API is simpler and more widely used, while the Responses API is more flexible and allows for richer interactions, including multi-turn conversations, tool usage, and structured outputs.

response = openai_client.responses.create(
    model="gpt-4.1-mini",
    input=prompt
)

In [42]:
response.output_text

'Yes, you can join the course now. You can start learning, watch the videos, work through the materials, and submit homework while submissions are still open. However, if you want to receive a certificate, make sure to submit your project before the submission deadline. Keep in mind that certificates are only awarded if you participate in a "live" cohort and complete the peer review process during the course running time.'

In [ ]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_0e5a34b1b0c76bc8006a3789b821e88197832fdf5f369fa60c",
  "created_at": 1782024632.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-4.1-mini-2025-04-14",
  "object": "response",
  "output": [
    {
      "id": "msg_0e5a34b1b0c76bc8006a3789b8b4988197a27e93ad7e7832f9",
      "content": [
        {
          "annotations": [],
          "text": "Yes, you can join the course now. You can start learning, watch the videos, work through the materials, and submit homework while submissions are still open. However, if you want to receive a certificate, make sure to submit your project before the submission deadline. Keep in mind that certificates are only awarded if you participate in a \"live\" cohort and complete the peer review process during the course running time.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "messag

In [ ]:
# Check the usage statistics of the API call

response.usage

ResponseUsage(input_tokens=491, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=83, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=574)

In [ ]:
# Calculate the cost of the API call based on the number of input and output tokens used. The pricing for gpt-4.1-mini is $0.40 per 1M tokens for input and $1.60 per 1M tokens for output.

input_price = 0.40 / 1_000_000  # $0.40 per 1M tokens for gpt-4.1-mini
output_price = 1.60 / 1_000_000  # $1.60 per 1M tokens for gpt-4.1-mini

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00032920000000000003

In [ ]:
# typical RAG pipeline as an abstraction / example

def rag(question):
    search_results = search(question) # Step 1: Search for relevant documents based on the question
    user_prompt = build_prompt(question, search_results) # Step 2: Build the RAG prompt by combining the question with the provided context
    return llm(user_prompt) # Step 3: Ask the model and return the answer